# Suspiciousness Scoring System

Комбінування всіх unsupervised методів для створення єдиної метрики підозрілості

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

## 1. Завантаження всіх результатів

In [ ]:
# Базовий датасет з anomaly detection
df = pd.read_csv('../../data/reviews_with_anomalies.csv')
print(f"Основний датасет: {len(df):,} записів")

# Кластерні дані (для відгуків з текстом)
df_clusters = pd.read_csv('../../data/reviews_with_clusters.csv')
print(f"Дані з кластерами: {len(df_clusters):,} записів")

# Similarity features (якщо є)
try:
    df_similarity = pd.read_csv('../../data/similarity_features.csv')
    print(f"Similarity features: {len(df_similarity):,} записів")
except:
    df_similarity = None
    print("Similarity features не знайдено (це нормально)")

## 2. Об'єднання всіх даних

In [ ]:
# Merge cluster data
df = df.merge(
    df_clusters[['cluster', 'outlier_score', 'umap_x', 'umap_y']], 
    left_index=True, 
    right_index=True, 
    how='left'
)

# Merge similarity data якщо є
if df_similarity is not None:
    df_similarity_indexed = df_similarity.set_index('index')
    df = df.merge(
        df_similarity_indexed[['avg_similarity', 'max_similarity']], 
        left_index=True, 
        right_index=True, 
        how='left'
    )

print(f"\nОб'єднаний датасет: {df.shape}")
print(f"Колонок: {len(df.columns)}")

## 3. Створення компонентів Suspiciousness Score

### 3.1. Cluster-based suspiciousness

In [ ]:
# Нормалізуємо outlier_score від HDBSCAN (0-1, де 1 = найбільш підозрілий)
scaler = MinMaxScaler()

# Для відгуків без тексту (NaN) - мінімальний score
df['cluster_suspicion'] = 0.0  # Default для відгуків без кластеру

mask_with_clusters = df['outlier_score'].notna()
if mask_with_clusters.sum() > 0:
    df.loc[mask_with_clusters, 'cluster_suspicion'] = scaler.fit_transform(
        df.loc[mask_with_clusters, 'outlier_score'].values.reshape(-1, 1)
    ).flatten()

# Якщо це outlier (кластер -1) - додаємо бонус
df.loc[df['cluster'] == -1, 'cluster_suspicion'] = df.loc[df['cluster'] == -1, 'cluster_suspicion'] * 1.3
df['cluster_suspicion'] = df['cluster_suspicion'].clip(0, 1)

print("Cluster suspicion створено")
print(f"Діапазон: {df['cluster_suspicion'].min():.3f} - {df['cluster_suspicion'].max():.3f}")

### 3.2. Anomaly-based suspiciousness

In [ ]:
# Нормалізуємо anomaly_score від Isolation Forest
df['anomaly_suspicion'] = scaler.fit_transform(
    df['anomaly_score'].values.reshape(-1, 1)
).flatten()

print("Anomaly suspicion створено")
print(f"Діапазон: {df['anomaly_suspicion'].min():.3f} - {df['anomaly_suspicion'].max():.3f}")

### 3.3. Text quality suspiciousness

In [ ]:
df['text_quality_suspicion'] = 0.0

# Дуже короткі шаблонні фрази (< 20 символів)
df.loc[(df['is_empty'] == 0) & (df['text_length'] < 20), 'text_quality_suspicion'] += 0.6

# Низька lexical diversity при наявності тексту (< 0.4) - шаблонні
df.loc[(df['is_empty'] == 0) & (df['unique_words_ratio'] < 0.4), 'text_quality_suspicion'] += 0.5

# Дуже багато великих літер (> 50%) - SPAM
df.loc[df['uppercase_ratio'] > 0.5, 'text_quality_suspicion'] += 0.4

# Багато цифр у тексті (підозріло для відгуку)
df.loc[df['digit_count'] > 10, 'text_quality_suspicion'] += 0.3

# Нормалізуємо до 0-1
df['text_quality_suspicion'] = df['text_quality_suspicion'].clip(0, 1)

print("Text quality suspicion створено")
print(f"Діапазон: {df['text_quality_suspicion'].min():.3f} - {df['text_quality_suspicion'].max():.3f}")
print(f"Відгуків з підозрілим текстом: {(df['text_quality_suspicion'] > 0).sum():,}")

### 3.4. Behavioral suspiciousness

In [ ]:
df['behavioral_suspicion'] = 0.0

# Review bursts - багато відгуків в один день (якщо є дані)
if 'reviews_on_day' in df.columns:
    burst_threshold = df['reviews_on_day'].quantile(0.95)
    df.loc[df['reviews_on_day'] > burst_threshold, 'behavioral_suspicion'] += 0.5

# Високий відсоток порожніх у лікаря (> 60%)
df.loc[df['doctor_empty_ratio'] > 0.6, 'behavioral_suspicion'] += 0.4

# Високий відсоток анонімних у лікаря (> 70%)
df.loc[df['doctor_anonymous_ratio'] > 0.7, 'behavioral_suspicion'] += 0.3

# Анонімний відгук
df.loc[df['is_anonymous'] == 1, 'behavioral_suspicion'] += 0.2

# Нормалізуємо до 0-1
df['behavioral_suspicion'] = df['behavioral_suspicion'].clip(0, 1)

print("Behavioral suspicion створено")
print(f"Діапазон: {df['behavioral_suspicion'].min():.3f} - {df['behavioral_suspicion'].max():.3f}")

### 3.5. Similarity-based suspiciousness

In [ ]:
# Підозріло якщо відгук дуже схожий на інші (копіпаста)
if 'max_similarity' in df.columns:
    df['similarity_suspicion'] = 0.0
    
    # Дуже висока схожість (> 0.95) підозріла
    df.loc[df['max_similarity'] > 0.95, 'similarity_suspicion'] = 0.8
    
    # Висока схожість (> 0.85)
    df.loc[(df['max_similarity'] > 0.85) & (df['max_similarity'] <= 0.95), 'similarity_suspicion'] = 0.5
    
    print("Similarity suspicion створено")
    print(f"Діапазон: {df['similarity_suspicion'].min():.3f} - {df['similarity_suspicion'].max():.3f}")
else:
    df['similarity_suspicion'] = 0.0
    print("Similarity suspicion пропущено (немає даних)")

## 4. Комбінований Suspiciousness Score

In [ ]:
# Weighted average - більше ваги на ML методи
weights = {
    'cluster_suspicion': 0.30,      # ↑ HDBSCAN outliers
    'anomaly_suspicion': 0.30,      # ↑ Isolation Forest
    'text_quality_suspicion': 0.10, # ↓ тільки для незвичайних паттернів
    'behavioral_suspicion': 0.25,   # ↑ bursts, анонімність
    'similarity_suspicion': 0.05    # копіпаста
}

df['suspiciousness_score'] = (
    df['cluster_suspicion'] * weights['cluster_suspicion'] +
    df['anomaly_suspicion'] * weights['anomaly_suspicion'] +
    df['text_quality_suspicion'] * weights['text_quality_suspicion'] +
    df['behavioral_suspicion'] * weights['behavioral_suspicion'] +
    df['similarity_suspicion'] * weights['similarity_suspicion']
)

print("\nSuspiciousness Score створено!")
print(f"Діапазон: {df['suspiciousness_score'].min():.3f} - {df['suspiciousness_score'].max():.3f}")
print(f"Середнє: {df['suspiciousness_score'].mean():.3f}")
print(f"Медіана: {df['suspiciousness_score'].median():.3f}")

## 5. Статистика та перцентилі

In [ ]:
print("\nСтатистика Suspiciousness Score:")
print(df['suspiciousness_score'].describe())

percentiles = [90, 95, 99]
print("\nПерцентилі:")
for p in percentiles:
    value = df['suspiciousness_score'].quantile(p/100)
    count = (df['suspiciousness_score'] >= value).sum()
    print(f"{p}th: {value:.3f} ({count:,} відгуків, {count/len(df)*100:.2f}%)")

## 6. Топ підозрілих - З ТЕКСТОМ (не порожні!)

**Шукаємо СПРАВДІ підозрілі відгуки з контентом**

In [ ]:
# Топ-100 підозрілих З ТЕКСТОМ
df_with_text = df[df['is_empty'] == 0].copy()
top_suspicious_with_text = df_with_text.nlargest(100, 'suspiciousness_score')[[
    'Коментар', "Ім'я лікаря", "Ім'я коментатора", 'Дата коментаря',
    'suspiciousness_score', 'cluster_suspicion', 'anomaly_suspicion',
    'text_quality_suspicion', 'behavioral_suspicion',
    'text_length', 'word_count', 'is_anonymous', 'unique_words_ratio'
]]

print("\n" + "="*100)
print("ТОП-20 ПІДОЗРІЛИХ ВІДГУКІВ З ТЕКСТОМ")
print("="*100)
print(top_suspicious_with_text[['Коментар', "Ім'я лікаря", 'suspiciousness_score', 'text_length']].head(20))

In [ ]:
# Детальний аналіз топ-10 з текстом
print("\n" + "="*100)
print("ДЕТАЛЬНИЙ АНАЛІЗ ТОП-10 ПІДОЗРІЛИХ З ТЕКСТОМ")
print("="*100)

doctor_name_col = "Ім'я лікаря"
commentator_name_col = "Ім'я коментатора"

for i, (idx, row) in enumerate(top_suspicious_with_text.head(10).iterrows(), 1):
    print(f"\n{i}. Score: {row['suspiciousness_score']:.4f}")
    print(f"   Компоненти: Cluster={row['cluster_suspicion']:.3f}, Anomaly={row['anomaly_suspicion']:.3f}, "
          f"Text={row['text_quality_suspicion']:.3f}, Behavioral={row['behavioral_suspicion']:.3f}")
    print(f"   Лікар: {row[doctor_name_col]}")
    print(f"   Коментатор: {row[commentator_name_col]}")
    print(f"   Дата: {row['Дата коментаря']}")
    print(f"   Статистика: {row['text_length']} символів, {row['word_count']} слів, "
          f"diversity={row['unique_words_ratio']:.2f}")
    print(f"   Анонімний: {'Так' if row['is_anonymous'] else 'Ні'}")
    text_preview = row['Коментар'][:200] if len(row['Коментар']) > 200 else row['Коментар']
    print(f"   Текст: {text_preview}...")
    print("-" * 100)

## 7. Аналіз порожніх відгуків окремо

Порожні відгуки - це окрема проблема (review padding), не класичні фейки

In [ ]:
empty_reviews = df[df['is_empty'] == 1].copy()

print(f"\nПорожніх відгуків: {len(empty_reviews):,} ({len(empty_reviews)/len(df)*100:.2f}%)")
print(f"\nТоп-10 лікарів з найбільшою кількістю порожніх відгуків:")
empty_by_doctor = empty_reviews["Ім'я лікаря"].value_counts().head(10)
print(empty_by_doctor)

# Які лікарі мають > 70% порожніх
doctor_stats = df.groupby("Ім'я лікаря").agg({
    'is_empty': ['sum', 'count', 'mean']
}).round(2)
doctor_stats.columns = ['empty_count', 'total', 'empty_ratio']
suspicious_doctors = doctor_stats[doctor_stats['empty_ratio'] > 0.7].sort_values('total', ascending=False)

print(f"\nЛікарів з > 70% порожніх відгуків: {len(suspicious_doctors)}")
print("\nТоп-10:")
print(suspicious_doctors.head(10))

## 8. Категоризація

In [ ]:
def categorize_suspiciousness(score):
    if score >= 0.7:
        return 'Дуже підозрілий'
    elif score >= 0.5:
        return 'Підозрілий'
    elif score >= 0.3:
        return 'Помірно підозрілий'
    else:
        return 'Нормальний'

df['suspiciousness_category'] = df['suspiciousness_score'].apply(categorize_suspiciousness)

category_counts = df['suspiciousness_category'].value_counts()
print("\nРозподіл за категоріями:")
for category, count in category_counts.items():
    print(f"{category}: {count:,} ({count/len(df)*100:.2f}%)")

# Окремо для відгуків з текстом - беремо з основного df з фільтром
print("\nДля відгуків З ТЕКСТОМ:")
category_counts_text = df[df['is_empty'] == 0]['suspiciousness_category'].value_counts()
for category, count in category_counts_text.items():
    print(f"{category}: {count:,} ({count/len(df_with_text)*100:.2f}%)")

## 9. Візуалізації

In [ ]:
# Розподіл scores
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].hist(df['suspiciousness_score'], bins=50, edgecolor='black', alpha=0.7)
axes[0, 0].set_xlabel('Suspiciousness Score')
axes[0, 0].set_ylabel('Частота')
axes[0, 0].set_title('Розподіл: ВСІ відгуки')
axes[0, 0].axvline(df['suspiciousness_score'].quantile(0.95), color='red', linestyle='--', label='95%')
axes[0, 0].legend()

axes[0, 1].hist(df_with_text['suspiciousness_score'], bins=50, edgecolor='black', alpha=0.7, color='green')
axes[0, 1].set_xlabel('Suspiciousness Score')
axes[0, 1].set_ylabel('Частота')
axes[0, 1].set_title('Розподіл: З ТЕКСТОМ')
axes[0, 1].axvline(df_with_text['suspiciousness_score'].quantile(0.95), color='red', linestyle='--', label='95%')
axes[0, 1].legend()

# Порівняння порожні vs з текстом
axes[1, 0].hist([df[df['is_empty']==1]['suspiciousness_score'], 
                 df[df['is_empty']==0]['suspiciousness_score']], 
                bins=30, label=['Порожні', 'З текстом'], alpha=0.6)
axes[1, 0].set_xlabel('Suspiciousness Score')
axes[1, 0].set_ylabel('Частота')
axes[1, 0].set_title('Порівняння: Порожні vs З текстом')
axes[1, 0].legend()

# Boxplot
axes[1, 1].boxplot([df[df['is_empty']==0]['suspiciousness_score'],
                    df[df['is_empty']==1]['suspiciousness_score']],
                   labels=['З текстом', 'Порожні'], vert=False)
axes[1, 1].set_xlabel('Suspiciousness Score')
axes[1, 1].set_title('Boxplot: З текстом vs Порожні')

plt.tight_layout()
plt.savefig('../../data/suspiciousness_score_distribution.png', dpi=150)
plt.show()

In [ ]:
# Категорії
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

category_order = ['Нормальний', 'Помірно підозрілий', 'Підозрілий', 'Дуже підозрілий']
colors = ['green', 'yellow', 'orange', 'red']

# Всі відгуки
category_counts_ordered = [category_counts.get(cat, 0) for cat in category_order]
axes[0].pie(category_counts_ordered, labels=category_order, autopct='%1.1f%%',
            colors=colors, startangle=90)
axes[0].set_title('Всі відгуки')

# З текстом
category_counts_text_ordered = [category_counts_text.get(cat, 0) for cat in category_order]
axes[1].pie(category_counts_text_ordered, labels=category_order, autopct='%1.1f%%',
            colors=colors, startangle=90)
axes[1].set_title('Тільки з текстом')

plt.tight_layout()
plt.savefig('../../data/suspiciousness_categories.png', dpi=150)
plt.show()

In [ ]:
# UMAP візуалізація (тільки для відгуків з текстом)
df_with_umap = df[(df['umap_x'].notna()) & (df['is_empty'] == 0)].copy()

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Колоруємо за score
scatter1 = axes[0].scatter(
    df_with_umap['umap_x'], df_with_umap['umap_y'], 
    c=df_with_umap['suspiciousness_score'],
    cmap='RdYlGn_r', alpha=0.5, s=10
)
axes[0].set_xlabel('UMAP 1')
axes[0].set_ylabel('UMAP 2')
axes[0].set_title('UMAP: Suspiciousness Score (тільки з текстом)')
plt.colorbar(scatter1, ax=axes[0], label='Score')

# Топ 5% підозрілих
very_suspicious = df_with_umap['suspiciousness_score'] >= df_with_umap['suspiciousness_score'].quantile(0.95)
axes[1].scatter(
    df_with_umap[~very_suspicious]['umap_x'], df_with_umap[~very_suspicious]['umap_y'], 
    c='lightgray', alpha=0.3, s=5, label='Нормальні'
)
axes[1].scatter(
    df_with_umap[very_suspicious]['umap_x'], df_with_umap[very_suspicious]['umap_y'], 
    c='red', alpha=0.7, s=30, label='Топ 5%'
)
axes[1].set_xlabel('UMAP 1')
axes[1].set_ylabel('UMAP 2')
axes[1].set_title('Топ 5% підозрілих (з текстом)')
axes[1].legend()

plt.tight_layout()
plt.savefig('../../data/umap_suspiciousness_visualization.png', dpi=150)
plt.show()

## 10. Збереження результатів

In [ ]:
# Повний датасет
df.to_csv('../../data/reviews_with_suspiciousness.csv', index=False)
print(f"Повний датасет: {len(df):,} записів")

# Топ підозрілих З ТЕКСТОМ
top_suspicious_with_text.to_csv('../../data/top_100_suspicious_reviews.csv', index=False)
print(f"Топ-100 з текстом збережено")

# Окремо порожні для аналізу review padding
empty_reviews.to_csv('../../data/empty_reviews_analysis.csv', index=False)
print(f"Порожні відгуки: {len(empty_reviews):,}")

# Summary
summary = pd.DataFrame({
    'Метрика': [
        'Всього відгуків',
        'З текстом',
        'Порожніх',
        'Середній score (всі)',
        'Середній score (з текстом)',
        '95-й перцентиль (всі)',
        '95-й перцентиль (з текстом)',
        'Підозрілих (>0.5, всі)',
        'Підозрілих (>0.5, з текстом)'
    ],
    'Значення': [
        len(df),
        len(df_with_text),
        len(empty_reviews),
        df['suspiciousness_score'].mean(),
        df_with_text['suspiciousness_score'].mean(),
        df['suspiciousness_score'].quantile(0.95),
        df_with_text['suspiciousness_score'].quantile(0.95),
        (df['suspiciousness_score'] >= 0.5).sum(),
        (df_with_text['suspiciousness_score'] >= 0.5).sum()
    ]
})

summary.to_csv('../../data/suspiciousness_summary.csv', index=False)
print("\nSummary:")
print(summary)